In [ ]:
import datetime
import gc

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import regularizers # it turned out that regularizers hampered the fitting...
from sklearn.model_selection import train_test_split

In [ ]:
plt.rcParams.update({
    "text.usetex": True, "font.family": "serif",
})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['savefig.dpi'] = 300

In [ ]:
test_dataframe = pd.read_csv('./data/burner_data.csv')

In [ ]:
x_data = test_dataframe[["t", "x_b", "q_squared", "phi"]]
y_data = test_dataframe[["unp_beam_unp_target_xsec", "unp_target_bsa"]]

In [ ]:
x_data.head()

In [ ]:
y_data.head()

In [ ]:
x_remaining, x_testing, y_remaining, y_testing = train_test_split(
    x_data, y_data,
    test_size = 0.1, shuffle = True)

x_training, x_validation, y_training, y_validation = train_test_split(
    x_remaining, y_remaining,
    test_size = 0.1, shuffle = True)

In [ ]:
len(x_training)

In [ ]:
len(x_validation)

In [ ]:
len(x_testing)

In [ ]:
def cff_h_model():
    # initializer:
    initializer = tf.keras.initializers.GlorotNormal(seed = None)

    # regularizer:
    # I learned about regularizers here: https://medium.com/@theo.wolf/physics-informed-neural-networks-a-simple-tutorial-with-pytorch-f28a890b874a
    # weight_regularizer = regularizers.l2(0.01) 
    
    # input layer:
    model_inputs = tf.keras.Input(shape = (4,), name = "input_values")

    # hidden layers:
    hidden = tf.keras.layers.Dense(
        48, kernel_initializer = initializer, activation = "tanh")(model_inputs)
    hidden = tf.keras.layers.Dense(
        48, kernel_initializer = initializer, activation = "tanh")(hidden)
    hidden = tf.keras.layers.Dense(
        48, kernel_initializer = initializer, activation = "tanh")(hidden)

    # output layer:
    model_output = tf.keras.layers.Dense(2, activation = "linear")(hidden)

    model = tf.keras.Model(inputs = model_inputs, outputs = model_output)

    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01),
        loss = tf.keras.losses.MeanSquaredError())
    
    return model

In [ ]:
tf.keras.backend.clear_session()
gc.collect()

dnn_model = cff_h_model()

dnn_model_history = dnn_model.fit(
    x_training, y_training,
    validation_data = (x_validation, y_validation),
    epochs = 1000, 
    # [NOTE]: BATCHSIZE really matters!
    batch_size = len(x_training),
    verbose = 1)

In [ ]:
model_testing_loss = dnn_model.evaluate(x_testing, y_testing, verbose = 0)
print(f"[INFO]: Test Loss: {model_testing_loss}")

In [ ]:
figure, axis = plt.subplots(1, 1, figsize = (6, 6))

axis.plot(dnn_model_history.history['loss'], 
    label = "Training Loss", color = 'orange', alpha = 0.6)
axis.plot(dnn_model_history.history['val_loss'], 
    label = "Validation Loss", color = 'purple', alpha = 0.6)

axis.set_xlabel("Epoch", fontsize = 14.)
axis.set_ylabel("Loss Values", fontsize = 14.)
axis.set_title(rf"Surrogate Model (Testing = ${model_testing_loss:.3f}$)",
    fontsize = 14.)
axis.legend(fontsize = 14.)
axis.grid(visible = False)

axis.text(
    0.00, -0.11,
    f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = axis.transAxes,
    verticalalignment = 'top',
    horizontalalignment = 'left',
    fontsize = 9.,
)

for extension in ['png', 'eps']:
    figure.savefig(
        fname = f"./plots/simultaneous_surrogate_v1.{extension}",
        facecolor = 'white',
        transparent = False)

plt.close(figure)

In [ ]:
grouped = test_dataframe.groupby(['t', 'x_b', 'q_squared'])

In [ ]:
phi_smooth = np.linspace(-np.pi, np.pi, 361) 

for (t_val, xb_val, q2_val), group in grouped:
    print(f"Processing t = {t_val}, xb = {xb_val}, Q2 = {q2_val}")

    group = group.sort_values('phi')
    x_group = group[['t', 'x_b', 'q_squared', 'phi']].values
    x_smooth = np.column_stack([
        np.full_like(phi_smooth, t_val),
        np.full_like(phi_smooth, xb_val),
        np.full_like(phi_smooth, q2_val),
        phi_smooth
    ])

    preds = dnn_model.predict(x_group)
    xsec_pred = preds[:, 0]
    bsa_pred  = preds[:, 1]

    smooth_preds = dnn_model.predict(x_smooth)
    xsec_smooth = smooth_preds[:, 0]
    bsa_smooth  = smooth_preds[:, 1]

    phi = group['phi'].values
    xsec_actual = group['unp_beam_unp_target_xsec'].values
    bsa_actual  = group['unp_target_bsa'].values

    xsec_res = xsec_actual - xsec_pred
    bsa_res  = bsa_actual - bsa_pred

    chi2_xsec = np.sum(xsec_res**2) / len(phi)
    chi2_bsa  = np.sum(bsa_res**2) / len(phi)

    residuals_figure, axes = plt.subplots(2, 2, figsize = (10, 8), sharex = 'col')

    axes[0, 0].scatter(phi, xsec_actual, color='black', alpha=0.7, label='Data')
    axes[0, 0].plot(phi_smooth, xsec_smooth, color='red', lw=2, label='DNN (interpolation)')
    axes[0, 0].set_ylabel(r"$d^{4}\sigma$", fontsize=14)
    axes[0, 0].set_title(rf"Cross Section ($\chi^2_\nu = {chi2_xsec:.3f}$)")
    axes[0, 0].legend()
    axes[0, 0].grid(True, linestyle=':', alpha=0.6)

    axes[0, 1].scatter(phi, xsec_res, color='blue', alpha=0.6)
    axes[0, 1].axhline(0, color='black', linestyle='--')
    axes[0, 1].set_title("Residuals")
    axes[0, 1].grid(True, linestyle=':', alpha=0.6)

    axes[1, 0].scatter(phi, bsa_actual, color='black', alpha=0.7, label='Data')
    axes[1, 0].plot(phi_smooth, bsa_smooth, color='green', lw=2, label='DNN (interpolation)')
    axes[1, 0].set_ylabel("BSA", fontsize=14)
    axes[1, 0].set_xlabel(r"$\phi$ (radians)", fontsize=14)
    axes[1, 0].set_title(rf"BSA ($\chi^2_\nu = {chi2_bsa:.3f}$)")
    axes[1, 0].legend()
    axes[1, 0].grid(True, linestyle=':', alpha=0.6)

    axes[1, 1].scatter(phi, bsa_res, color='purple', alpha=0.6)
    axes[1, 1].axhline(0, color='black', linestyle='--')
    axes[1, 1].set_xlabel(r"$\phi$ (radians)", fontsize=14)
    axes[1, 1].set_title("Residuals")
    axes[1, 1].grid(True, linestyle=':', alpha=0.6)
    
    residuals_figure.suptitle(
        rf"$t = {t_val:.3f}$, $x_\textrm{{B}} = {xb_val:.3f}$, $Q^2={q2_val:.3f}$",
        fontsize=16
    )

    filename = f"./plots/t{t_val:.3f}_xb{xb_val:.3f}_q2{q2_val:.3f}_residuals_v1"
    for extension in ['png', 'eps']:
        residuals_figure.savefig(
            fname = f"{filename}.{extension}",
            facecolor = 'white',
            transparent = False)

    plt.close(residuals_figure)